# ACE-Net Baseline — Results & Visualization

Reproduces the headline tables of **ACE-Net** (Yu et al., *Electronics* 2025,
14, 4420) from the checkpoints trained by this repo's pipeline, and compares
them against the values reported in the paper.

This notebook **loads the baseline's own code** (`src/`) — the same model,
data, split, and metric functions used in training — so the numbers here are
exactly what `eval_stage1.py` / `eval_stage2.py` produce.

**Sections**
1. Setup & environment
2. Checkpoint inventory
3. Table 2 — Speech–Text emotion recognition
4. Table 3 — Facial emotion recognition
5. Confusion matrices (Figures 5/6)
6. Table 4 — Forgery detection by pairing type
7. Comparison vs paper

> If a checkpoint is missing, its section will say so and skip — train it
> first with the commands in the repo README, then re-run that cell.

## 1. Setup & environment

Add the repo root to `sys.path`, import the baseline modules, pick the device.

In [ ]:
import sys, os
from pathlib import Path

# repo root = parent of this notebook's folder
ROOT = Path.cwd()
if (ROOT / "src").exists():
    REPO = ROOT
elif (ROOT.parent / "src").exists():
    REPO = ROOT.parent
else:
    raise RuntimeError("run this notebook from the repo root or notebooks/")
sys.path.insert(0, str(REPO))
print("repo:", REPO)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

from src import config
from src.config import TrainConfig
from src.data import manifests
from src.train_utils import set_seed, stratified_split

cfg = TrainConfig()
set_seed(cfg.seed)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CKPT = config.CKPT_ROOT
print("device:", DEVICE, "| checkpoints dir:", CKPT)

## 2. Checkpoint inventory

Which trained models are available. Stage-1 produces one checkpoint per
(branch, dataset); Stage-2 produces the full ACE-Net.

In [ ]:
def ckpt_status():
    rows = []
    expected = {
        "stage1_speech_text_crema.pt": "Stage-1 speech-text (CREMA-D)",
        "stage1_visual_crema.pt":      "Stage-1 visual (CREMA-D)",
        "stage1_speech_text_meld.pt":  "Stage-1 speech-text (MELD)",
        "stage1_visual_meld.pt":       "Stage-1 visual (MELD)",
        "stage2_acenet.pt":            "Stage-2 ACE-Net (fusion+MLP)",
    }
    for fn, desc in expected.items():
        p = CKPT / fn
        rows.append({"checkpoint": fn, "description": desc,
                     "present": "yes" if p.exists() else "MISSING"})
    return pd.DataFrame(rows)

inv = ckpt_status()
display(inv)
HAVE = {r["checkpoint"]: (r["present"] == "yes") for _, r in inv.iterrows()}

## 3. Table 2 — Speech–Text Emotion Recognition

The MDCNN + cross-attention branch, evaluated on the held-out test split
(same seeded 80/10/10 split as training — no leakage). Reports **Accuracy**
and **Weighted-F1** (the paper's primary emotion metric).

In [ ]:
from src.models.speech_text import SpeechTextModule
from src.models.fv_litenet import FVLiteNet
from src.eval_stage1 import _collect_preds, metrics as emo_metrics, confusion as emo_confusion
from src.data.dataset import EmotionDataset, collate_emotion
from torch.utils.data import DataLoader


def eval_stage1(branch, dataset, batch_size=16):
    """Return (y_true, y_pred, class_names) for the held-out test split."""
    tag = f"{branch}_{dataset}"
    ckpt = CKPT / f"stage1_{tag}.pt"
    if not ckpt.exists():
        return None
    names = config.EMOTION_DATASETS[dataset]["emotions"]
    nc = len(names)
    samples = manifests.build_emotion_samples(dataset)
    _, _, test_s = stratified_split(
        samples, key_fn=lambda s: s.emotion,
        ratios=(cfg.train_ratio, cfg.val_ratio, cfg.test_ratio), seed=cfg.seed)
    model = (SpeechTextModule(n_classes=nc) if branch == "speech_text"
             else FVLiteNet(n_classes=nc)).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    dl = DataLoader(EmotionDataset(test_s), batch_size=batch_size,
                    shuffle=False, num_workers=0, collate_fn=collate_emotion)
    y, p = _collect_preds(model, dl, branch, DEVICE)
    return y, p, names

In [ ]:
def emotion_table(branch, datasets=("crema", "meld")):
    rows = []
    cache = {}
    for ds in datasets:
        res = eval_stage1(branch, ds)
        if res is None:
            rows.append({"Dataset": ds.upper(), "ACC %": None, "Weighted-F1": None,
                         "n": 0, "note": "checkpoint missing"})
            continue
        y, p, names = res
        acc, wf1 = emo_metrics(y, p, len(names))
        rows.append({"Dataset": ds.upper(), "ACC %": round(acc*100, 2),
                     "Weighted-F1": round(wf1, 3), "n": len(y), "note": ""})
        cache[ds] = (y, p, names)
    return pd.DataFrame(rows), cache

st_tbl, st_cache = emotion_table("speech_text")
print("Table 2 — Speech-Text Emotion Recognition (ACE-Net = MDCNN row)")
display(st_tbl)

In [ ]:
def bar_emotion(df, title):
    d = df.dropna(subset=["ACC %"])
    if d.empty:
        print("no trained checkpoints yet for:", title); return
    fig, ax = plt.subplots(figsize=(6, 4))
    x = np.arange(len(d)); w = 0.35
    ax.bar(x - w/2, d["ACC %"], w, label="ACC %", color="#4C72B0")
    ax.bar(x + w/2, d["Weighted-F1"]*100, w, label="Weighted-F1 (x100)", color="#DD8452")
    ax.set_xticks(x); ax.set_xticklabels(d["Dataset"])
    ax.set_ylabel("score"); ax.set_title(title); ax.legend(); ax.set_ylim(0, 100)
    for i, (a, f) in enumerate(zip(d["ACC %"], d["Weighted-F1"]*100)):
        ax.text(i - w/2, a + 1, f"{a:.1f}", ha="center", fontsize=8)
        ax.text(i + w/2, f + 1, f"{f:.1f}", ha="center", fontsize=8)
    plt.tight_layout(); plt.show()

bar_emotion(st_tbl, "Table 2 — Speech-Text Emotion (MDCNN)")

## 4. Table 3 — Facial Emotion Recognition

The FV-LiteNet (GhostNet-based) visual branch, same protocol.

In [ ]:
vis_tbl, vis_cache = emotion_table("visual")
print("Table 3 — Facial Emotion Recognition (ACE-Net = FV-LiteNet row)")
display(vis_tbl)
bar_emotion(vis_tbl, "Table 3 — Facial Emotion (FV-LiteNet)")

## 5. Confusion Matrices (Figures 5/6)

Per-class confusion on the held-out test set, row-normalized. Strong diagonal
= good per-class discrimination.

In [ ]:
def plot_confusion(cache, title_prefix):
    if not cache:
        print("no checkpoints for", title_prefix); return
    n = len(cache)
    fig, axes = plt.subplots(1, n, figsize=(6*n, 5))
    if n == 1:
        axes = [axes]
    for ax, (ds, (y, p, names)) in zip(axes, cache.items()):
        cm = emo_confusion(y, p, len(names)).astype(float)
        cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
        sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues", cbar=False,
                    xticklabels=names, yticklabels=names, ax=ax, vmin=0, vmax=1)
        ax.set_title(f"{title_prefix} — {ds.upper()}")
        ax.set_xlabel("predicted"); ax.set_ylabel("true")
    plt.tight_layout(); plt.show()

plot_confusion(st_cache, "Speech-Text")
plot_confusion(vis_cache, "Facial")

## 6. Table 4 — Forgery Detection by Pairing Type

The full ACE-Net (frozen extractors + fusion + MLP) on the held-out Stage-2
test split, broken down by pairing type:

- **Genuine Pairs** — aligned real audio+face (label 0)
- **Emotion Tampering** — P1: same identity, mismatched emotion (label 1)
- **Cross-Identity Spliced** — P2: different identity + emotion (label 1)

Forgery rows are scored one-vs-genuine, matching the paper's per-type table:
ACC / Precision / Recall / F1 / AUC.

In [ ]:
from src.models.acenet import ACENet
from src.eval_stage2 import (build_balanced_tagged, collect as s2_collect,
                             auc_score, bin_metrics, dataset_of)


def eval_stage2(batch_size=16):
    ckpt = CKPT / "stage2_acenet.pt"
    if not ckpt.exists():
        return None
    samples = build_balanced_tagged(cfg.seed)
    _, _, test_s = stratified_split(
        samples, key_fn=lambda s: s.label,
        ratios=(cfg.train_ratio, cfg.val_ratio, cfg.test_ratio), seed=cfg.seed)
    model = ACENet().to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.eval()
    y, p = s2_collect(model, test_s, DEVICE, batch_size, 0)
    types = np.array([s.ptype for s in test_s])
    dsets = np.array([dataset_of(s) for s in test_s])
    return y, p, types, dsets


s2 = eval_stage2()
if s2 is None:
    print("stage2_acenet.pt missing — train Stage-2 first.")
    table4 = pd.DataFrame()
else:
    y, p, types, dsets = s2
    rows = []
    for ds in sorted(set(dsets.tolist())):
        ds_gen = (dsets == ds) & (y == 0)
        for pt in ["Genuine Pairs", "Emotion Tampering", "Cross-Identity Spliced"]:
            sel = (dsets == ds) & (types == pt)
            if sel.sum() == 0:
                continue
            if pt == "Genuine Pairs":
                acc = float((p[sel] < 0.5).mean())
                rows.append({"Dataset": ds, "Pairing Type": pt,
                             "ACC %": round(acc*100, 1), "Prec": None,
                             "Recall": None, "F1": None, "AUC": None})
            else:
                idx = sel | ds_gen
                acc, pr, rc, f1 = bin_metrics(y[idx], p[idx])
                rows.append({"Dataset": ds, "Pairing Type": pt,
                             "ACC %": round(acc*100, 1), "Prec": round(pr, 3),
                             "Recall": round(rc, 3), "F1": round(f1, 3),
                             "AUC": round(auc_score(y[idx], p[idx]), 3)})
    table4 = pd.DataFrame(rows)
    display(table4)

In [ ]:
if not table4.empty:
    d = table4.dropna(subset=["AUC"]).copy()
    if not d.empty:
        fig, ax = plt.subplots(figsize=(8, 4.5))
        labels = [f"{r['Dataset']}\n{r['Pairing Type'].split()[0]}" for _, r in d.iterrows()]
        x = np.arange(len(d))
        ax.bar(x, d["AUC"], 0.5, color=["#55A868" if "Cross" in t else "#C44E52"
                                        for t in d["Pairing Type"]])
        ax.axhline(0.5, ls="--", c="gray", lw=1, label="chance")
        ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
        ax.set_ylabel("AUC"); ax.set_ylim(0, 1.0)
        ax.set_title("Table 4 — Forgery detection AUC by type")
        for i, v in enumerate(d["AUC"]):
            ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=8)
        ax.legend(); plt.tight_layout(); plt.show()

## 7. Comparison vs Paper

Side-by-side of our reproduced numbers against the values reported in the
paper. Gaps are expected where data differs (genuine LastHalf only, no SAVEE,
6GB GPU vs 24GB). See `FIDELITY.md`.

In [ ]:
# paper-reported reference values (from docs/ACE-Net.md tables)
PAPER = {
    "Table2_MDCNN": {"CREMA": 74.67, "MELD": 68.92},        # ACC %
    "Table3_FVLiteNet": {"CREMA": 86.83, "MELD": 71.77},    # ACC %
    "Table4_CREMA_AUC": {"Emotion Tampering": 0.92,
                          "Cross-Identity Spliced": 0.94},
    "Table4_MELD_AUC": {"Emotion Tampering": 0.77,
                         "Cross-Identity Spliced": 0.80},
}


def compare_emotion(our_df, paper_key, model_name):
    rows = []
    for _, r in our_df.iterrows():
        ds = r["Dataset"]
        paper_acc = PAPER[paper_key].get("CREMA" if ds == "CREMA" else
                                          "MELD" if ds == "MELD" else ds)
        ours = r["ACC %"]
        rows.append({"Dataset": ds, "Model": model_name,
                     "Ours ACC %": ours, "Paper ACC %": paper_acc,
                     "Δ": None if ours is None else round(ours - paper_acc, 2)})
    return pd.DataFrame(rows)

cmp = pd.concat([
    compare_emotion(st_tbl, "Table2_MDCNN", "MDCNN (speech-text)"),
    compare_emotion(vis_tbl, "Table3_FVLiteNet", "FV-LiteNet (visual)"),
], ignore_index=True)
print("Emotion recognition: ours vs paper")
display(cmp)

In [ ]:
d = cmp.dropna(subset=["Ours ACC %"]).copy()
if not d.empty:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(d)); w = 0.35
    ax.bar(x - w/2, d["Ours ACC %"], w, label="Ours", color="#4C72B0")
    ax.bar(x + w/2, d["Paper ACC %"], w, label="Paper", color="#8172B3")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{r.Dataset}\n{r.Model.split()[0]}" for r in d.itertuples()],
                       fontsize=8)
    ax.set_ylabel("ACC %"); ax.set_ylim(0, 100)
    ax.set_title("Emotion recognition: reproduced vs paper")
    ax.legend(); plt.tight_layout(); plt.show()
else:
    print("train Stage-1 checkpoints to populate this comparison.")

---

### Notes
- All evaluations use the **held-out test split** re-derived with the same
  seed as training — never seen during fitting.
- "MISSING" sections mean that checkpoint has not been trained yet. Train with:
  ```
  python -m src.train_stage1 --branch speech_text --dataset crema
  python -m src.train_stage1 --branch visual      --dataset crema
  python -m src.train_stage2
  ```
  then re-run the notebook.
- Not reproducible here: **SAVEE** rows (no data) and **Table 6 / DFDC**
  (no DFDC; FakeAVCeleb is raw and needs preprocessing).